# Beyond Words — Phase 5: ASR front-ends on Ben-10 valid (Kaggle)

Transcribes the public Ben-10 validation set (1,666 clips with human dialect transcripts)
with every front-end, so `ben10_eval.py` can compute WER/CER and dialect-word recall locally.

**Notebook settings:** GPU T4, Internet ON. No dataset input is needed: the audio (about 1 GB)
is fetched from the Hugging Face dataset `bengaliAI/Ben-10` (CC0).

In [ ]:
!git clone --depth 1 https://github.com/towfique-elahe/beyond-words.git /tmp/beyond-words
!nvidia-smi -L

In [ ]:
import pandas as pd
from huggingface_hub import snapshot_download

ROOT = snapshot_download('bengaliAI/Ben-10', repo_type='dataset',
                         allow_patterns=['valid/*'], local_dir='/tmp/ben10')
ref = pd.read_csv(f'{ROOT}/valid/valid.csv')
manifest = pd.DataFrame({'relpath': 'valid/' + ref['file_name'], 'district': ref['district']})
manifest.to_csv('/tmp/ben10_manifest.csv', index=False)
print(len(manifest), 'clips')

MODELS = [
    ('tugstugi_ben10', 'bengaliAI/tugstugi_bengaliai-regional-asr_whisper-medium', 'whisper',
     '--gen-config-from openai/whisper-medium'),
    ('tugstugi', 'bengaliAI/tugstugi_bengaliai-asr_whisper-medium', 'whisper',
     '--gen-config-from openai/whisper-medium'),
    ('whisper_large_v3', 'openai/whisper-large-v3', 'whisper', ''),
    ('w2v_xlsr_bn', 'arijitx/wav2vec2-xls-r-300m-bengali', 'ctc', '--batch 4'),
    ('hishab_fastconformer', 'hishab/hishab_bn_fastconformer', 'nemo', ''),
]

In [ ]:
import time

for tag, model, backend, extra in MODELS:
    if backend == 'nemo':
        !pip install -q "nemo_toolkit[asr]"
    t0 = time.time()
    !python /tmp/beyond-words/transcribe.py \
        --model {model} --backend {backend} --tag {tag} {extra} \
        --manifest /tmp/ben10_manifest.csv --audio-root {ROOT} --out-dir /kaggle/working
    print(f'{tag}: {(time.time() - t0) / 60:.1f} min')

In [ ]:
from pathlib import Path

for tag, *_ in MODELS:
    f = Path('/kaggle/working') / f'{tag}.csv'
    print(tag, len(pd.read_csv(f)) if f.exists() else 'MISSING')

Download the CSVs into the local repo's `transcripts/ben10_valid/` and run
`python ben10_eval.py transcripts/ben10_valid/*.csv`.